In [1]:
import cv2
import numpy as np
import time
from scipy.signal import butter, filtfilt, find_peaks
from collections import deque

In [25]:
# -----------------------------
# CAMERA SETUP
# -----------------------------
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not found")
    exit()

In [26]:
# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [27]:
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
cap.set(cv2.CAP_PROP_EXPOSURE, 0)

True

In [28]:
# -----------------------------
# SIGNAL STORAGE
# -----------------------------
signal_buffer = []
time_buffer = []

BUFFER_SIZE = 300

start_time = time.time()

In [29]:
# -----------------------------
# FILTER FUNCTION
# -----------------------------
def bandpass_filter(signal, fs):
    low = 0.8
    high = 3.0

    nyquist = 0.5 * fs

    low /= nyquist
    high /= nyquist

    b, a = butter(3, [low, high], btype='band')

    return filtfilt(b, a, signal)

In [30]:
# -----------------------------
# BPM CALCULATION
# -----------------------------
def calculate_bpm(signal, timestamps):

    if len(signal) < 100:
        return None

    duration = timestamps[-1] - timestamps[0]

    if duration <= 0:
        return None

    fs = len(signal) / duration

    try:
        filtered = bandpass_filter(signal, fs)

        peaks, _ = find_peaks(filtered, distance=fs/2)

        bpm = (len(peaks) / duration) * 60

        return int(bpm)

    except:
        return None

In [31]:
# -----------------------------
# MAIN LOOP
# -----------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(120, 120)
    )

    bpm = None

    for (x, y, w, h) in faces:

        # Face box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        # Forehead ROI
        fx = x + int(w * 0.3)
        fy = y + int(h * 0.15)
        fw = int(w * 0.4)
        fh = int(h * 0.15)

        cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), (255,0,0), 2)

        roi = frame[fy:fy+fh, fx:fx+fw]

        if roi.size > 0:

            green_mean = np.mean(roi[:, :, 1])

            current_time = time.time()

            signal_buffer.append(green_mean)
            time_buffer.append(current_time)

            if len(signal_buffer) > BUFFER_SIZE:
                signal_buffer.pop(0)
                time_buffer.pop(0)

            bpm = calculate_bpm(signal_buffer, time_buffer)

        break

    # -----------------------------
    # DISPLAY PANEL
    # -----------------------------
    cv2.rectangle(frame, (20,20), (350,180), (30,30,30), -1)

    cv2.putText(
        frame,
        "AI CAMERA VITAL MONITOR",
        (35,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,255),
        2
    )

    if bpm is not None and 40 < bpm < 180:

        # VERY rough BP estimation
        est_sys = int(110 + (bpm - 60) * 0.5)
        est_dia = int(70 + (bpm - 60) * 0.3)

        cv2.putText(
            frame,
            f"BPM: {bpm}",
            (35,90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Estimated BP: {est_sys}/{est_dia}",
            (35,130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,255),
            2
        )

    else:

        cv2.putText(
            frame,
            "Detecting pulse...",
            (35,100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,165,255),
            2
        )

    # -----------------------------
    # DRAW PULSE GRAPH
    # -----------------------------
    if len(signal_buffer) > 50:

        graph_x = 20
        graph_y = 220
        graph_w = 300
        graph_h = 100

        cv2.rectangle(
            frame,
            (graph_x, graph_y),
            (graph_x + graph_w, graph_y + graph_h),
            (50,50,50),
            -1
        )

        normalized = np.array(signal_buffer)

        normalized = normalized - np.min(normalized)

        if np.max(normalized) != 0:
            normalized = normalized / np.max(normalized)

        points = []

        for i, val in enumerate(normalized[-graph_w:]):

            px = graph_x + i
            py = graph_y + graph_h - int(val * graph_h)

            points.append((px, py))

        for i in range(1, len(points)):
            cv2.line(
                frame,
                points[i-1],
                points[i],
                (0,255,0),
                1
            )

    # -----------------------------
    # SHOW OUTPUT
    # -----------------------------
    cv2.imshow("Camera BP & BPM Monitor", frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

KeyboardInterrupt: 

In [32]:
cap.release()
cv2.destroyAllWindows()

In [2]:
# =========================================================
# CAMERA SETUP
# =========================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not found")
    exit()

# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Camera settings
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
cap.set(cv2.CAP_PROP_EXPOSURE, 0)

# =========================================================
# GLOBAL VARIABLES
# =========================================================

signal_buffer = []
time_buffer = []

BUFFER_SIZE = 300

smoothed_bpm = None

peak_times = deque(maxlen=30)

stress_level = "LOW"

fatigue_score = 0

signal_quality = 0

wellness_score = 100

# =========================================================
# FILTER FUNCTION
# =========================================================

def bandpass_filter(signal, fs):

    low = 0.8
    high = 3.0

    nyquist = 0.5 * fs

    low /= nyquist
    high /= nyquist

    b, a = butter(3, [low, high], btype='band')

    return filtfilt(b, a, signal)

# =========================================================
# BPM SMOOTHING
# =========================================================

def smooth_bpm(new_bpm):

    global smoothed_bpm

    alpha = 0.15

    if smoothed_bpm is None:
        smoothed_bpm = new_bpm
    else:
        smoothed_bpm = (
            alpha * new_bpm
            + (1 - alpha) * smoothed_bpm
        )

    return int(smoothed_bpm)

# =========================================================
# HRV CALCULATION
# =========================================================

def calculate_hrv(peaks, timestamps):

    if len(peaks) < 3:
        return 0

    beat_times = [timestamps[p] for p in peaks]

    intervals = np.diff(beat_times)

    if len(intervals) < 2:
        return 0

    hrv = np.std(intervals) * 1000

    return int(hrv)

# =========================================================
# BPM CALCULATION
# =========================================================

def calculate_bpm(signal, timestamps):

    if len(signal) < 100:
        return None

    duration = timestamps[-1] - timestamps[0]

    if duration <= 0:
        return None

    fs = len(signal) / duration

    try:

        signal = np.array(signal)

        # Normalize signal
        signal = (
            signal - np.mean(signal)
        ) / np.std(signal)

        filtered = bandpass_filter(signal, fs)

        peaks, _ = find_peaks(
            filtered,
            distance=fs/2
        )

        bpm = (len(peaks) / duration) * 60

        return int(bpm), peaks, filtered

    except:
        return None

# =========================================================
# MAIN LOOP
# =========================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(120, 120)
    )

    bpm = None
    hrv = 0
    breathing_rate = 0
    filtered_signal = None
    peaks = []

    # =====================================================
    # FACE PROCESSING
    # =====================================================

    for (x, y, w, h) in faces:

        # Face box
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0,255,0),
            2
        )

        # Forehead ROI
        fx = x + int(w * 0.3)
        fy = y + int(h * 0.15)
        fw = int(w * 0.4)
        fh = int(h * 0.15)

        cv2.rectangle(
            frame,
            (fx, fy),
            (fx+fw, fy+fh),
            (255,0,0),
            2
        )

        roi = frame[
            fy:fy+fh,
            fx:fx+fw
        ]

        if roi.size > 0:

            # Extract green channel
            green_mean = np.mean(
                roi[:, :, 1]
            )

            signal_buffer.append(green_mean)

            time_buffer.append(time.time())

            # Keep buffer fixed size
            if len(signal_buffer) > BUFFER_SIZE:

                signal_buffer.pop(0)
                time_buffer.pop(0)

            result = calculate_bpm(
                signal_buffer,
                time_buffer
            )

            if result:

                bpm, peaks, filtered_signal = result

                bpm = smooth_bpm(bpm)

        break

    # =====================================================
    # HEALTH ANALYTICS
    # =====================================================

    if bpm is not None and 40 < bpm < 180:

        # HRV
        if len(peaks) > 2:

            hrv = calculate_hrv(
                peaks,
                time_buffer
            )

        # Stress detection
        if bpm > 95 or hrv < 40:
            stress_level = "HIGH"

        elif bpm > 80 or hrv < 60:
            stress_level = "MEDIUM"

        else:
            stress_level = "LOW"

        # Breathing estimation
        if filtered_signal is not None:

            breathing_signal = np.convolve(
                filtered_signal,
                np.ones(15)/15,
                mode='same'
            )

            breath_peaks, _ = find_peaks(
                breathing_signal,
                distance=20
            )

            duration = (
                time_buffer[-1]
                - time_buffer[0]
            )

            if duration > 0:

                breathing_rate = int(
                    (len(breath_peaks)
                    / duration) * 60
                )

        # Signal quality
        variance = np.var(signal_buffer)

        signal_quality = min(
            100,
            int(variance * 40)
        )

        # Fatigue
        if bpm < 55:
            fatigue_score += 1
        else:
            fatigue_score = max(
                0,
                fatigue_score - 1
            )

        fatigue_status = (
            "TIRED"
            if fatigue_score > 20
            else "ACTIVE"
        )

        # Wellness score
        wellness_score = 100

        if bpm > 100:
            wellness_score -= 20

        if hrv < 40:
            wellness_score -= 25

        if breathing_rate > 22:
            wellness_score -= 15

        if signal_quality < 20:
            wellness_score -= 10

        wellness_score = max(
            0,
            wellness_score
        )

        # BP estimation
        est_sys = int(
            110 + (bpm - 60) * 0.5
        )

        est_dia = int(
            70 + (bpm - 60) * 0.3
        )

    # =====================================================
    # UI PANEL
    # =====================================================

    cv2.rectangle(
        frame,
        (20,20),
        (460,360),
        (30,30,30),
        -1
    )

    cv2.putText(
        frame,
        "AI WELLNESS MONITOR",
        (35,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,255,255),
        2
    )

    if bpm is not None and 40 < bpm < 180:

        cv2.putText(
            frame,
            f"BPM: {bpm}",
            (35,90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Estimated BP: {est_sys}/{est_dia}",
            (35,125),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,255),
            2
        )

        cv2.putText(
            frame,
            f"HRV: {hrv} ms",
            (35,160),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,255),
            2
        )

        cv2.putText(
            frame,
            f"Stress: {stress_level}",
            (35,195),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,165,255),
            2
        )

        cv2.putText(
            frame,
            f"Breathing: {breathing_rate}/min",
            (35,230),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,255),
            2
        )

        cv2.putText(
            frame,
            f"Fatigue: {fatigue_status}",
            (35,265),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,255),
            2
        )

        cv2.putText(
            frame,
            f"Signal Quality: {signal_quality}%",
            (35,300),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,255),
            2
        )

        cv2.putText(
            frame,
            f"Wellness Score: {wellness_score}",
            (35,335),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    else:

        cv2.putText(
            frame,
            "Detecting pulse...",
            (35,100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,165,255),
            2
        )

    # =====================================================
    # PULSE GRAPH
    # =====================================================

    if len(signal_buffer) > 50:

        graph_x = 500
        graph_y = 40

        graph_w = 350
        graph_h = 200

        cv2.rectangle(
            frame,
            (graph_x, graph_y),
            (graph_x+graph_w,
             graph_y+graph_h),
            (50,50,50),
            -1
        )

        normalized = np.array(signal_buffer)

        normalized = (
            normalized
            - np.min(normalized)
        )

        if np.max(normalized) != 0:

            normalized = (
                normalized
                / np.max(normalized)
            )

        points = []

        for i, val in enumerate(
            normalized[-graph_w:]
        ):

            px = graph_x + i

            py = (
                graph_y
                + graph_h
                - int(val * graph_h)
            )

            points.append((px, py))

        for i in range(1, len(points)):

            cv2.line(
                frame,
                points[i-1],
                points[i],
                (0,255,0),
                1
            )

        cv2.putText(
            frame,
            "Pulse Signal",
            (graph_x, graph_y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            2
        )

    # =====================================================
    # SHOW WINDOW
    # =====================================================

    cv2.imshow(
        "AI Camera Wellness Monitor",
        frame
    )

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

# =========================================================
# CLEANUP
# =========================================================

cap.release()
cv2.destroyAllWindows()

In [1]:
pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.


In [3]:
cap.release()
cv2.destroyAllWindows()